# 🎨 محوّل الصور الفني - اليوم الوطني
# AI Style Transfer - National Day

---

## مرحباً بكم في تجربة التحويل الفني! 🇱🇾

### كيفية الاستخدام | How to Use:

1. **ارفع صورتك** أو التقط صورة بالكاميرا | Upload your photo or take a webcam shot
2. **اختر النمط الفني** المفضل لديك | Choose your preferred artistic style
3. **اضغط على زر التحويل** وانتظر النتيجة | Click Transform and wait for the result
4. **حمّل الصورة** الفنية الجديدة | Download your new artistic image

### الأنماط المتاحة | Available Styles:
- 🌌 **أنمي / كرتون** | Anime / Cartoon
- 🎨 **لوحة مائية** | Watercolor Painting
- 🖼️ **لوحة زيتية** | Oil Painting
- ✏️ **رسم بالرصاص** | Pencil Sketch
- 🇱🇾 **النمط الوطني** | National Day Theme

---
> **ملاحظة:** قد تستغرق عملية التحويل من 10 إلى 30 ثانية حسب النمط المختار
> **Note:** Processing may take 10-30 seconds depending on the chosen style

In [ ]:
# Cell 1: Environment Check + Install Maximum Quality Stack

import sys, subprocess, torch
print(f'Python {sys.version.split()[0]} | PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('  ⚠ No GPU — Runtime > Change runtime type > T4 GPU')

print('\nInstalling maximum quality pipeline packages...')

# Core SD pipeline
subprocess.run(['pip','install','-q','diffusers>=0.27.0','transformers','accelerate','xformers'], check=False)

# Face identity: InsightFace (face embedding) + IP-Adapter FaceID
subprocess.run(['pip','install','-q','insightface','onnxruntime-gpu'], check=False)
subprocess.run(['pip','install','-q','git+https://github.com/tencent-ailab/IP-Adapter.git'], check=False)

# Post-processing: GFPGAN (face restoration) + Real-ESRGAN (4x upscale)
subprocess.run(['pip','install','-q','basicsr','facexlib','gfpgan','realesrgan'], check=False)

# UI
subprocess.run(['pip','install','-q','gradio'], check=False)
subprocess.run(['pip','install','-q','qrcode[pil]'], check=False)

print('\n✓ All packages installed — proceed to Cell 2')

In [ ]:
# Cell 2: Imports & Global Setup

import os, gc, sys, time, warnings
import cv2, numpy as np, torch
import gradio as gr
from PIL import Image, ImageEnhance
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    StableDiffusionControlNetImg2ImgPipeline,
    StableDiffusionImg2ImgPipeline,
    DPMSolverMultistepScheduler,
)
from huggingface_hub import hf_hub_download
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE == 'cuda' else torch.float32

print('=' * 60)
print('  Maximum Quality AI Style Transfer Pipeline')
print('  InsightFace → IP-Adapter FaceID → Counterfeit-V3')
print('  → GFPGAN face restore → Real-ESRGAN x2 upscale')
print('=' * 60)
print(f'Device : {DEVICE.upper()}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Global model handles (all lazy-loaded once)
_face_app        = None   # InsightFace — face embedding
_faceid_pipe     = None   # SD pipeline used by IPAdapterFaceID
_ip_faceid_model = None   # IPAdapterFaceID wrapper
_controlnet_pipe = None   # ControlNet img2img (fallback path)
_sd_pipe         = None   # Base SD 1.5 (watercolor/oil)
_restorer        = None   # GFPGAN
_upsampler       = None   # Real-ESRGAN anime
_USE_FACEID      = False  # set True after FaceID loads successfully

MODEL_DIR = '/content/models'
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# Cell 3: Download Weights & Load All Models

# ── 1. InsightFace face analyser ─────────────────────────
def load_face_app():
    global _face_app
    if _face_app: return _face_app
    print('[1/4] Loading InsightFace...')
    try:
        import insightface
        from insightface.app import FaceAnalysis
        _face_app = FaceAnalysis(
            name='buffalo_l',
            providers=['CUDAExecutionProvider','CPUExecutionProvider'])
        _face_app.prepare(ctx_id=0, det_size=(640,640))
        print('  InsightFace ready!')
    except Exception as e:
        print(f'  InsightFace failed: {e}')
        _face_app = None
    return _face_app


# ── 2. ONE pipeline: Counterfeit-V3.0 + ControlNet ───────
# Then we TRY to attach FaceID on top.
# If FaceID is unavailable, the pipeline still works standalone.
def load_main_pipeline():
    global _faceid_pipe, _ip_faceid_model, _USE_FACEID

    if _faceid_pipe:
        return _faceid_pipe

    print('[2/4] Loading ControlNet canny (~1.4 GB)...')
    cn = ControlNetModel.from_pretrained(
        'lllyasviel/sd-controlnet-canny', torch_dtype=DTYPE)

    # Anime models tried in order — first one that exists on HuggingFace is used
    ANIME_MODELS = [
        'dreamlike-art/dreamlike-anime-1.0',   # official, actively maintained
        'andite/anything-v4.0',                # popular anime model
        'runwayml/stable-diffusion-v1-5',      # base model (always exists)
    ]
    loaded = False
    for model_id in ANIME_MODELS:
        try:
            print(f'[2/4] Loading {model_id}...')
            _faceid_pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
                model_id,
                controlnet=cn,
                torch_dtype=DTYPE,
                safety_checker=None,
                requires_safety_checker=False,
            )
            print(f'  Loaded: {model_id}')
            loaded = True
            break
        except Exception as e:
            print(f'  {model_id} failed ({type(e).__name__}) — trying next...')
    if not loaded:
        raise RuntimeError('All anime models failed to load')
    _faceid_pipe.scheduler = DPMSolverMultistepScheduler.from_config(
        _faceid_pipe.scheduler.config)
    _faceid_pipe = _faceid_pipe.to(DEVICE)

    if DEVICE == 'cuda':
        _faceid_pipe.enable_attention_slicing()
        _faceid_pipe.enable_vae_slicing()
        try: _faceid_pipe.enable_xformers_memory_efficient_attention()
        except: pass
        used  = torch.cuda.memory_allocated()/1e9
        total = torch.cuda.get_device_properties(0).total_memory/1e9
        print(f'  VRAM used: {used:.1f}/{total:.1f} GB')

    print('  Base pipeline ready! Trying FaceID...')

    # Download FaceID weights
    ckpt = f'{MODEL_DIR}/ip-adapter-faceid_sd15.bin'
    if not os.path.exists(ckpt):
        print('  Downloading FaceID weights (~100 MB)...')
        try:
            hf_hub_download('h94/IP-Adapter-FaceID',
                            'ip-adapter-faceid_sd15.bin',
                            local_dir=MODEL_DIR)
        except Exception as e:
            print(f'  FaceID download failed: {e}')

    if os.path.exists(ckpt):
        try:
            from ip_adapter.ip_adapter_faceid import IPAdapterFaceID
            _ip_faceid_model = IPAdapterFaceID(_faceid_pipe, ckpt, DEVICE)
            _USE_FACEID = True
            print('  IP-Adapter FaceID attached — face identity LOCKED!')
        except Exception as e:
            print(f'  FaceID attach failed: {e}')
            print('  Pipeline works WITHOUT FaceID (face identity not locked)')
            _USE_FACEID = False
    else:
        print('  FaceID weights missing — running without FaceID')
        _USE_FACEID = False

    return _faceid_pipe


# ── 3. GFPGAN face restorer ───────────────────────────────
def load_restorer():
    global _restorer
    if _restorer: return _restorer
    gfpgan_path = f'{MODEL_DIR}/GFPGANv1.4.pth'
    if not os.path.exists(gfpgan_path):
        print('[3/4] Downloading GFPGAN v1.4 (~330 MB)...')
        import urllib.request
        try:
            urllib.request.urlretrieve(
                'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth',
                gfpgan_path)
        except Exception as e:
            print(f'  GFPGAN download failed: {e}'); return None
    print('[3/4] Loading GFPGAN...')
    try:
        from gfpgan import GFPGANer
        _restorer = GFPGANer(
            model_path=gfpgan_path, upscale=1,
            arch='clean', channel_multiplier=2, bg_upsampler=None)
        print('  GFPGAN ready!')
    except Exception as e:
        print(f'  GFPGAN failed: {e}'); _restorer = None
    return _restorer


# ── 4. Real-ESRGAN anime upscaler ────────────────────────
def load_upsampler():
    global _upsampler
    if _upsampler: return _upsampler
    path = f'{MODEL_DIR}/RealESRGAN_x4plus_anime_6B.pth'
    if not os.path.exists(path):
        print('[4/4] Downloading Real-ESRGAN anime (~17 MB)...')
        import urllib.request
        try:
            urllib.request.urlretrieve(
                'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth',
                path)
        except Exception as e:
            print(f'  ESRGAN download failed: {e}'); return None
    print('[4/4] Loading Real-ESRGAN anime...')
    try:
        from realesrgan import RealESRGANer
        from basicsr.archs.rrdbnet_arch import RRDBNet
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                        num_block=6, num_grow_ch=32, scale=4)
        _upsampler = RealESRGANer(
            scale=4, model_path=path, model=model,
            tile=512, tile_pad=10, pre_pad=0,
            half=(DEVICE == 'cuda'))
        print('  Real-ESRGAN ready!')
    except Exception as e:
        print(f'  Real-ESRGAN failed: {e}'); _upsampler = None
    return _upsampler


# ── Load everything ───────────────────────────────────────
print('Loading models — first run takes 5-10 min...\n')
load_face_app()
load_main_pipeline()
load_restorer()
load_upsampler()

print(f'\n{"="*55}')
print(f'Face embedding  : {"InsightFace + FaceID ✓" if _USE_FACEID else "basic (FaceID unavailable)"}')
print(f'Style model     : Counterfeit-V3.0 + ControlNet canny')
print(f'Face restore    : {"GFPGAN v1.4 ✓" if _restorer else "skipped"}')
print(f'Upscale         : {"Real-ESRGAN anime 2x ✓" if _upsampler else "skipped"}')
print(f'{"="*55}')

In [ ]:
# Cell 4: Style Transfer Functions

# ── Image helpers ─────────────────────────────────────────
def pil_to_cv2(img):
    return cv2.cvtColor(np.array(img.convert('RGB')), cv2.COLOR_RGB2BGR)

def cv2_to_pil(img):
    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

def resize_to_sd(img, target=512):
    w, h = img.size
    nw, nh = (target, int(h*target/w)) if w>=h else (int(w*target/h), target)
    return img.resize((max(nw//8*8, 64), max(nh//8*8, 64)), Image.LANCZOS)

def resize_keep_aspect(img, max_side=768):
    w, h = img.size
    if max(w, h) <= max_side: return img
    s = max_side / max(w, h)
    return img.resize((int(w*s)//8*8, int(h*s)//8*8), Image.LANCZOS)

def get_canny(pil_image, low=60, high=160):
    img = np.array(pil_image.convert('RGB'))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5,5), 0), low, high)
    return Image.fromarray(cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB))


# ── Seed & eye color helpers ────────────────────────────

def make_generator(seed=42):
    """Fixed seed → reproducible results. seed=-1 → random each time."""
    if seed == -1:
        seed = torch.randint(0, 2**32, (1,)).item()
    torch.manual_seed(seed)
    np.random.seed(seed % (2**31))
    return torch.Generator(device=DEVICE).manual_seed(seed), seed

def detect_eye_color(image):
    """
    Sample iris pixels at InsightFace eye keypoints.
    Filters for high-saturation pixels only (iris > sclera/skin).
    Returns prompt string like 'green eyes' or None.
    """
    app = load_face_app()
    if app is None: return None
    try:
        img_arr = np.array(image.convert('RGB'))
        faces   = app.get(img_arr)
        if not faces: return None
        face = max(faces, key=lambda f: f.det_score)
        kps  = face.kps            # [left_eye, right_eye, nose, ...]
        h, w = img_arr.shape[:2]
        r    = max(3, min(h, w) // 90)   # tight radius — iris only

        iris_colors = []
        for eye_pt in kps[:2]:
            x, y = int(eye_pt[0]), int(eye_pt[1])
            patch_rgb = img_arr[max(0,y-r):y+r, max(0,x-r):x+r]
            if patch_rgb.size == 0: continue
            # Keep only high-saturation pixels (iris, not white sclera or skin)
            patch_hsv = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2HSV)
            sat_mask  = patch_hsv[:, :, 1] > 35   # saturation threshold
            if sat_mask.sum() >= 3:
                iris_colors.append(patch_rgb[sat_mask].mean(0))
            else:
                iris_colors.append(patch_rgb.reshape(-1, 3).mean(0))

        if not iris_colors: return None
        rv, gv, bv = np.mean(iris_colors, axis=0)

        # Improved classification using hue distance
        if bv > max(rv, gv) + 12:   return 'blue eyes'
        if gv > rv + 8 and gv > bv + 8:  return 'green eyes'
        if gv > rv - 5 and rv > 80:  return 'hazel eyes'
        if rv > bv + 15:             return 'brown eyes'
        return 'dark brown eyes'
    except Exception as e:
        print(f'  Eye color detection error: {e}'); return None


def detect_hair_color(image):
    """Estimate hair color from the top portion of the image above the face."""
    app = load_face_app()
    if app is None: return None
    try:
        img_arr = np.array(image.convert('RGB'))
        faces   = app.get(img_arr)
        if not faces: return None
        face = max(faces, key=lambda f: f.det_score)
        x1, y1, x2, y2 = face.bbox.astype(int)
        # Sample the region just ABOVE the face bounding box (hair area)
        hair_y2 = max(0, y1)
        hair_y1 = max(0, y1 - (y2 - y1) // 2)
        hair_x1 = max(0, x1)
        hair_x2 = min(img_arr.shape[1], x2)
        patch = img_arr[hair_y1:hair_y2, hair_x1:hair_x2]
        if patch.size == 0: return None
        rv, gv, bv = patch.reshape(-1, 3).mean(0)
        # Simple hair color classification
        brightness = (rv + gv + bv) / 3
        if brightness > 200:           return 'blonde hair'
        if brightness > 150:           return 'light brown hair'
        if rv > gv + 15:               return 'dark brown hair'
        if brightness < 60:            return 'black hair'
        return 'brown hair'
    except Exception as e:
        print(f'  Hair color detection error: {e}'); return None


def preserve_background(original_pil, styled_pil):
    """
    Keep original background. Detect face on ORIGINAL image,
    then SCALE the bbox to styled image size — avoids det_size mismatch.
    """
    app = load_face_app()
    if app is None: return styled_pil
    try:
        ow, oh = original_pil.size
        sw, sh = styled_pil.size

        # Detect on ORIGINAL (reliable det_size match)
        orig_arr = np.array(original_pil.convert('RGB'))
        faces = app.get(orig_arr)
        if not faces:
            print('  Background: no face found, returning styled as-is')
            return styled_pil

        face = max(faces, key=lambda f: f.det_score)
        x1o, y1o, x2o, y2o = face.bbox.astype(int)

        # Scale bbox coords from original → styled resolution
        sx, sy = sw / ow, sh / oh
        x1, y1 = int(x1o*sx), int(y1o*sy)
        x2, y2 = int(x2o*sx), int(y2o*sy)

        face_h = y2 - y1
        face_w = x2 - x1
        cx, cy = (x1+x2)//2, (y1+y2)//2
        margin = max(face_h, face_w) * 1.8   # enough to cover hair+shoulders

        nx1 = max(0, int(cx - margin * 0.90))
        ny1 = max(0, int(cy - margin * 1.10))  # extra room for hair above
        nx2 = min(sw, int(cx + margin * 0.90))
        ny2 = min(sh, int(cy + margin * 1.20))  # shoulders below

        # Soft feathered mask
        mask = np.zeros((sh, sw), dtype=np.float32)
        mask[ny1:ny2, nx1:nx2] = 1.0
        blur_r = max(21, int(margin * 0.30)) | 1   # must be odd
        mask = cv2.GaussianBlur(mask, (blur_r, blur_r), 0)
        mask3 = mask[:, :, np.newaxis]

        # Original resized to styled size
        orig_rs  = np.array(
            original_pil.convert('RGB').resize((sw, sh), Image.LANCZOS),
            dtype=np.float32)
        styled_a = np.array(styled_pil.convert('RGB'), dtype=np.float32)

        out = (styled_a * mask3 + orig_rs * (1.0 - mask3)).clip(0,255).astype(np.uint8)
        print(f'  Background preserved (face mask {nx1},{ny1}→{nx2},{ny2})')
        return Image.fromarray(out)
    except Exception as e:
        print(f'  Background preserve error: {e}'); return styled_pil

# ── Face embedding ────────────────────────────────────────
def get_face_embedding(image):
    app = load_face_app()
    if app is None: return None
    try:
        faces = app.get(np.array(image.convert('RGB')))
        if not faces: return None
        best = max(faces, key=lambda f: f.det_score)
        return torch.from_numpy(best.normed_embedding).unsqueeze(0)
    except Exception as e:
        print(f'  Face embedding error: {e}'); return None


# ── Post-processing ───────────────────────────────────────
def restore_face(pil_image):
    r = load_restorer()
    if r is None: return pil_image
    try:
        _, _, restored = r.enhance(
            pil_to_cv2(pil_image),
            has_aligned=False, only_center_face=False, paste_back=True)
        return cv2_to_pil(restored)
    except Exception as e:
        print(f'  GFPGAN error: {e}'); return pil_image

def upscale_2x(pil_image):
    u = load_upsampler()
    if u is None: return pil_image
    try:
        output, _ = u.enhance(pil_to_cv2(pil_image), outscale=2)
        result = cv2_to_pil(output)
        # Mild desaturation: ESRGAN anime_6B over-enhances colors
        return ImageEnhance.Color(result).enhance(0.88)
    except Exception as e:
        print(f'  ESRGAN error: {e}'); return pil_image


# ── OpenCV guaranteed fallback ────────────────────────────
def _anime_opencv(image):
    img = pil_to_cv2(image)
    h, w = img.shape[:2]
    if max(h, w) > 800:
        s = 800/max(h,w); img = cv2.resize(img,(int(w*s)//2*2,int(h*s)//2*2))
    smooth = img.copy()
    for _ in range(8): smooth = cv2.bilateralFilter(smooth, 9, 75, 75)
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.adaptiveThreshold(cv2.medianBlur(gray,7), 255,
                cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 9, 2)
    res = cv2.bitwise_and(smooth, cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR))
    hsv = cv2.cvtColor(res, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:,:,1] = np.clip(hsv[:,:,1]*2.3, 0, 255)
    hsv[:,:,2] = np.clip(hsv[:,:,2]*1.1, 0, 255)
    out = cv2_to_pil(cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR))
    return ImageEnhance.Color(out.resize(image.size, Image.LANCZOS)).enhance(1.6)


# ── Prompts per style ─────────────────────────────────────
PROMPTS = {
    'kawaii': (
        "(masterpiece:1.4),(best quality:1.3),(ultra-detailed:1.2),"
        "kawaii anime girl,chibi style,big sparkling eyes,"
        "soft round face,smooth cel shading,soft pastel colors,warm natural lighting",
        "(worst quality:1.4),(low quality:1.4),realistic,photo,"
        "ugly,bad anatomy,deformed,extra limbs,text,watermark,nsfw,"
        "neon colors,oversaturated,psychedelic"
    ),
    'anime_hd': (
        "(masterpiece:1.3),(best quality:1.2),"
        "anime portrait,beautiful detailed eyes,clean sharp lineart,"
        "vibrant colors,cel shaded,professional anime illustration",
        "(worst quality:1.4),(low quality:1.4),realistic,photo,"
        "blurry,bad anatomy,deformed,text,watermark,nsfw"
    ),
    'cartoon': (
        "(masterpiece:1.3),(best quality:1.2),"
        "studio ghibli style,cartoon character,smooth rounded face,"
        "big expressive eyes,warm soft lighting,soft natural colors,natural skin tone",
        "(worst quality:1.4),(low quality:1.4),realistic,photo,"
        "blurry,bad anatomy,text,watermark,dark,nsfw,"
        "neon colors,oversaturated,psychedelic,rainbow hair"
    ),
    'cartoon_portrait': (
        "(masterpiece:1.4),(best quality:1.3),(ultra-detailed:1.2),"
        "western cartoon portrait,disney style,cartoon character,"
        "thick black outlines,big expressive eyes,detailed hair,"
        "smooth clean skin,natural skin tone,flat illustration,"
        "professional cartoon art,same outfit as original",
        "(worst quality:1.4),(low quality:1.4),anime,chibi,realistic,photo,"
        "3d render,ugly,bad anatomy,deformed,text,watermark,nsfw,blurry,"
        "neon colors,oversaturated,psychedelic,rainbow hair,teal hair,purple hair"
    ),
}


# ── Core generator — 3-level fallback chain ───────────────
def generate_anime(image, style_key='kawaii',
                    steps=30, guidance=9.0, cn_scale=0.70,
                    faceid_scale=0.8, img2img_strength=0.60,
                    seed=42):
    """
    Fallback chain — guarantees a result:
      A) img2img + ControlNet + FaceID  (max quality + identity preserved)
      B) img2img + ControlNet           (reliable identity via pixel content)
      C) AnimeGANv2 → OpenCV            (always works)

    seed: fixed → same result every run. -1 → different each time.
    Eye color: auto-detected from photo, injected into prompt.
    """
    img_512     = resize_to_sd(image.convert('RGB'), 512)
    prompt, neg = PROMPTS.get(style_key, PROMPTS['kawaii'])
    canny       = get_canny(img_512, 60, 160)
    generator, used_seed = make_generator(seed)
    print(f'  Seed: {used_seed}')

    # ── Auto-detect eye color → inject into prompt ──────────
    # Hair color removed: sampling above face is too noisy (background bleeds in)
    eye_color = detect_eye_color(img_512)
    if eye_color:
        print(f'  Eye color : {eye_color}')
        # Weight 1.2 — enough to guide without fighting the style
        prompt = f'({eye_color}:1.2), ' + prompt
        wrong_eyes = {'blue eyes','green eyes','brown eyes',
                      'hazel eyes','dark brown eyes'} - {eye_color}
        neg = ', '.join(wrong_eyes) + ', ' + neg
    else:
        print('  Eye color : not detected')

    ctx = torch.autocast(DEVICE) if DEVICE == 'cuda' else torch.no_grad()
    result = None

    # ── Level A: img2img + ControlNet + FaceID ────────────
    if _USE_FACEID and _ip_faceid_model is not None:
        print('  [A] img2img + ControlNet + FaceID...')
        try:
            faceid_embeds = get_face_embedding(img_512)
            if faceid_embeds is None:
                faceid_embeds = torch.zeros(1, 512)
            with ctx:
                images = _ip_faceid_model.generate(
                    faceid_embeds=faceid_embeds,
                    prompt=prompt, negative_prompt=neg,
                    scale=faceid_scale,
                    num_samples=1,
                    num_inference_steps=steps,
                    guidance_scale=guidance,
                    seed=used_seed,            # reproducible
                    image=img_512,
                    control_image=canny,
                    strength=img2img_strength,
                    controlnet_conditioning_scale=cn_scale,
                )
            result = images[0]
            print('  [A] Success!')
        except Exception as e:
            print(f'  [A] FaceID error: {e}')

    # ── Level B: img2img + ControlNet (no FaceID) ─────────
    # Starts from the real photo → same person's face features preserved
    if result is None and _faceid_pipe is not None:
        print('  [B] img2img + ControlNet (face from photo pixels)...')
        try:
            with ctx:
                result = _faceid_pipe(
                    prompt=prompt, negative_prompt=neg,
                    image=img_512,
                    control_image=canny,
                    strength=img2img_strength,
                    guidance_scale=guidance,
                    num_inference_steps=steps,
                    controlnet_conditioning_scale=cn_scale,
                    generator=generator,       # reproducible
                ).images[0]
            print('  [B] Success!')
        except Exception as e:
            print(f'  [B] img2img error: {e}')

    # ── Level C: AnimeGANv2 → OpenCV ──────────────────────
    if result is None:
        print('  [C] AnimeGAN/OpenCV guaranteed fallback...')
        result = anime_fast_style(image)

    if DEVICE == 'cuda':
        torch.cuda.empty_cache(); gc.collect()

    # ── Post-processing: GFPGAN → ESRGAN → background ──────
    print('  Post: GFPGAN face restore...')
    result = restore_face(result)
    print('  Post: Real-ESRGAN 2x upscale...')
    result = upscale_2x(result)
    print('  Post: Preserving original background...')
    # Pass ORIGINAL image — preserve_background handles scaling internally
    result = preserve_background(image, result)
    return result


# ── Style wrappers ────────────────────────────────────────
def kawaii_anime_style(image, strength=0.7, seed=42):
    return generate_anime(image, 'kawaii',
                          steps=30, guidance=7.0, cn_scale=0.70,
                          faceid_scale=0.85, img2img_strength=0.55, seed=seed)

def anime_hd_style(image, strength=0.65, seed=42):
    return generate_anime(image, 'anime_hd',
                          steps=28, guidance=7.0, cn_scale=0.65,
                          faceid_scale=0.80, img2img_strength=0.55, seed=seed)

def cartoon_style(image, strength=0.68, seed=42):
    return generate_anime(image, 'cartoon',
                          steps=28, guidance=7.0, cn_scale=0.75,
                          faceid_scale=0.75, img2img_strength=0.60, seed=seed)

def cartoon_portrait_style(image, strength=0.72, seed=42):
    """Western cartoon / Disney style — closest to @whincy_lab aesthetic."""
    return generate_anime(image, 'cartoon_portrait',
                          steps=32, guidance=7.5, cn_scale=0.65,
                          faceid_scale=0.90, img2img_strength=0.58, seed=seed)


# ── OpenCV-only styles (instant, no GPU model) ────────────
def watercolor_style(image, strength=0.6):
    img = pil_to_cv2(image)
    h,w = img.shape[:2]
    if max(h,w)>1024: s=1024/max(h,w); img=cv2.resize(img,(int(w*s),int(h*s)))
    out = cv2.stylization(img,sigma_s=int(60+strength*60),sigma_r=0.3+strength*0.25)
    out = cv2.addWeighted(out,0.7,cv2.GaussianBlur(out,(3,3),0),0.3,0)
    hsv = cv2.cvtColor(out,cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:,:,1] = np.clip(hsv[:,:,1]*1.3,0,255)
    return ImageEnhance.Color(cv2_to_pil(
        cv2.cvtColor(hsv.astype(np.uint8),cv2.COLOR_HSV2BGR))).enhance(1.15)

def oil_painting_style(image, strength=0.6):
    img = pil_to_cv2(image)
    h,w = img.shape[:2]
    if max(h,w)>1024: s=1024/max(h,w); img=cv2.resize(img,(int(w*s),int(h*s)))
    out = cv2.detailEnhance(img,sigma_s=int(10+strength*10),sigma_r=0.15)
    n = max(6,int(16-strength*8))
    _,lbl,ctr = cv2.kmeans(np.float32(out).reshape((-1,3)),n,None,
        (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER,20,0.5),5,
        cv2.KMEANS_RANDOM_CENTERS)
    quant = np.uint8(ctr)[lbl.flatten()].reshape(out.shape)
    blend = cv2.edgePreservingFilter(
        cv2.addWeighted(out,0.4,quant,0.6,0),flags=1,sigma_s=30,sigma_r=0.4)
    return ImageEnhance.Color(
        ImageEnhance.Contrast(cv2_to_pil(blend)).enhance(1.1)).enhance(1.2)

def pencil_sketch(image, strength=0.6):
    img = pil_to_cv2(image)
    h,w = img.shape[:2]
    if max(h,w)>1024: s=1024/max(h,w); img=cv2.resize(img,(int(w*s),int(h*s)))
    gray,_ = cv2.pencilSketch(img,sigma_s=60,sigma_r=0.07,
                               shade_factor=0.03+strength*0.05)
    return ImageEnhance.Contrast(
        cv2_to_pil(cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR))).enhance(1.3)

_animegan_model = None
def anime_fast_style(image, strength=0.55):
    global _animegan_model
    try:
        from torchvision.transforms.functional import to_tensor, to_pil_image
        if _animegan_model is None:
            _animegan_model = torch.hub.load(
                'bryandlee/animegan2-pytorch:main', 'generator',
                pretrained='face_paint_512_v2', force_reload=False)
            _animegan_model = _animegan_model.to(DEVICE).eval()
        inp = to_tensor(image.convert('RGB')).unsqueeze(0).to(DEVICE) * 2 - 1
        with torch.no_grad():
            out = _animegan_model(inp)[0]
        out_pil = to_pil_image((out.cpu().clamp(-1, 1) + 1) / 2)
        return out_pil.resize(image.size, Image.LANCZOS)
    except Exception as e:
        print(f'  AnimeGAN fallback: {e}')
        return _anime_opencv(image)


def national_day_style(image, strength=0.6):
    """
    Libyan National Day themed style with green color wash.
    Applies warm desaturation then tints with Libyan green (0x239e46).
    """
    img_cv = pil_to_cv2(image)
    h, w = img_cv.shape[:2]
    if max(h, w) > 1024:
        s = 1024 / max(h, w)
        img_cv = cv2.resize(img_cv, (int(w * s), int(h * s)))
    # Stylize
    stylized = cv2.stylization(img_cv, sigma_s=80, sigma_r=0.4)
    # Convert to HSV and apply green tint
    hsv = cv2.cvtColor(stylized, cv2.COLOR_BGR2HSV).astype(np.float32)
    # Desaturate slightly
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * 0.7, 0, 255)
    # Shift hue toward green (Libyan green hue ~75 in HSV 0-179)
    hsv[:, :, 0] = 75
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] + 30, 0, 255)
    result = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    # Blend with original for subtlety
    blended = cv2.addWeighted(img_cv, 1 - strength, result, strength, 0)
    return cv2_to_pil(blended)

print('Style functions defined! Proceed to Cell 5.')

In [ ]:
# Cell 5: Gradio UI

import os, tempfile, base64
import io as _io
import gradio as gr

# ── All available styles ──────────────────────────────────
STYLE_MAP = {
    'كيواي | Kawaii Anime'         : kawaii_anime_style,
    'أنمي HD | Anime HD'           : anime_hd_style,
    'كرتون | Cartoon (Ghibli)'     : cartoon_style,
    'كرتون غربي | Western Cartoon' : cartoon_portrait_style,
    'لوحة مائية | Watercolor'      : watercolor_style,
    'لوحة زيتية | Oil Painting'    : oil_painting_style,
    'رسم بالرصاص | Pencil Sketch'  : pencil_sketch,
    'النمط الوطني | National Day'  : national_day_style,
}

# ── Wallet-size helpers ────────────────────────────────────
def to_wallet_size(image):
    W, H = 750, 1050
    img = image.copy()
    tr = W / H
    ir = img.width / img.height
    if ir > tr:
        nw = int(img.height * tr)
        left = (img.width - nw) // 2
        img = img.crop((left, 0, left + nw, img.height))
    else:
        nh = int(img.width / tr)
        top = (img.height - nh) // 2
        img = img.crop((0, top, img.width, top + nh))
    return img.resize((W, H), Image.LANCZOS)

def _gradio_cache_dir():
    d = os.environ.get('GRADIO_TEMP_DIR', os.path.join(tempfile.gettempdir(), 'gradio'))
    os.makedirs(d, exist_ok=True)
    return d

def build_save_print_html(wallet_path):
    buf = _io.BytesIO()
    Image.open(wallet_path).save(buf, format='JPEG', quality=80)
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    img_data_url = 'data:image/jpeg;base64,' + img_b64

    iframe_src = (
        '<!DOCTYPE html><html><head><style>'
        'body{margin:0;display:flex;flex-direction:column;align-items:center;'
        'justify-content:center;height:100%;background:#fff;padding:4px;}'
        'img{width:110px;height:154px;object-fit:cover;border:1px solid #ddd;}'
        'button{margin-top:6px;background:#1d4ed8;color:#fff;border:none;'
        'padding:6px 14px;font-size:0.82em;border-radius:5px;cursor:pointer;}'
        '@media print{@page{size:2.5in 3.5in;margin:0;}'
        'body{margin:0;padding:0;}'
        'img{width:2.5in;height:3.5in;object-fit:cover;}'
        'button{display:none;}}'
        '</style></head>'
        '<body><img src="' + img_data_url + '">'
        '<button onclick="window.print()">&#128424; Print</button>'
        '</body></html>'
    )
    srcdoc = iframe_src.replace('&', '&amp;').replace('"', '&quot;')
    return (
        '<div style="text-align:center;padding:8px;">'
        '<a href="' + img_data_url + '" download="cartoon_wallet.jpg" style="'
        'display:inline-block;background:#16a34a;color:white;'
        'text-decoration:none;padding:10px 22px;'
        'font-size:0.95em;font-weight:bold;border-radius:8px;">'
        '📱 &nbsp; Save Photo | حفظ على الهاتف'
        '</a>'
        '<br><iframe srcdoc="' + srcdoc + '" '
        'style="margin-top:8px;width:150px;height:210px;border:none;'
        'border-radius:6px;box-shadow:0 1px 6px rgba(0,0,0,.15);">'
        '</iframe>'
        '<br><p style="font-size:0.72em;color:#888;margin:3px 0 0;">'
        '2.5&Prime; &times; 3.5&Prime; &nbsp;&bull;&nbsp; 300 DPI</p>'
        '</div>'
    )

# ── Main transform ────────────────────────────────────────
def transform_image(image, style_name, strength, seed=42):
    if image is None:
        return None, 'يرجى رفع صورة | Please upload a photo'
    fn = STYLE_MAP.get(style_name)
    if fn is None:
        return None, 'Unknown style: ' + style_name
    try:
        pil = image if isinstance(image, Image.Image) else Image.fromarray(image)
        result = fn(pil, strength, seed=seed)
        return result, 'تم التحويل | Done'
    except Exception as e:
        return None, 'Error: ' + str(e)

# ── Gradio UI ─────────────────────────────────────────────
with gr.Blocks(
    title='AI Style Transfer',
    theme=gr.themes.Soft(primary_hue='blue'),
) as demo:

    gr.Markdown('# AI Style Transfer | محوّل الصور الفني')

    with gr.Tabs():
        with gr.Tab('Webcam | كاميرا'):
            webcam_input = gr.Image(
                sources=['webcam'], type='pil',
                label='التقط صورة | Take a photo',
                mirror_webcam=True, height=300,
            )
            use_webcam_btn = gr.Button(
                'استخدم هذه الصورة | Use this photo',
                variant='secondary',
            )
        with gr.Tab('Upload | رفع صورة'):
            upload_input = gr.Image(
                sources=['upload', 'clipboard'], type='pil',
                label='ارفع من الجهاز | Upload from device',
                height=300,
            )

    working_image = gr.Image(
        type='pil',
        label='الصورة المختارة | Selected photo',
        height=180, interactive=False,
    )
    use_webcam_btn.click(fn=lambda x: x, inputs=[webcam_input], outputs=[working_image])
    upload_input.change(fn=lambda x: x, inputs=[upload_input], outputs=[working_image])

    with gr.Row():
        with gr.Column(scale=1):
            style_selector = gr.Radio(
                choices=list(STYLE_MAP.keys()),
                value=list(STYLE_MAP.keys())[0],
                label='النمط الفني | Art Style',
            )
            strength_slider = gr.Slider(
                minimum=0.30, maximum=0.80, value=0.55, step=0.05,
                label='Style Strength | قوة الاسلوب',
            )
            seed_input = gr.Number(
                value=42, precision=0,
                label='Seed (42=fixed, -1=random)',
            )
            transform_btn = gr.Button(
                'Transform | حول الان',
                variant='primary', size='lg',
            )

        with gr.Column(scale=1):
            output_image = gr.Image(
                label='Result | النتيجة',
                type='pil', height=320, interactive=False,
                elem_id='result-img',
            )
            status_box = gr.Textbox(
                label='Status | الحالة',
                lines=2, interactive=False,
            )
            download_btn = gr.DownloadButton(
                label='Download | تحميل',
                visible=False,
            )
            sp_box = gr.HTML(label='Save / Print')

    # QR panel — filled by demo.load JS (scripts here DO execute)
    gr.HTML(
        '<div id="qr-panel" style="text-align:center;padding:10px;">'
        '<p style="color:#aaa;font-size:0.82em;">'
        'QR يظهر بعد التحويل | QR appears after transform'
        '</p></div>'
    )

    def on_transform(image, style, strength, seed=42):
        result, status = transform_image(image, style, strength, seed=int(seed))
        if result is not None:
            wallet = to_wallet_size(result)
            wallet_path = os.path.join(_gradio_cache_dir(), 'styled_wallet.jpg')
            wallet.save(wallet_path, format='JPEG', dpi=(300, 300), quality=92)
            sp_html = build_save_print_html(wallet_path)
            return result, status, gr.update(value=wallet_path, visible=True), sp_html
        return None, status, gr.update(visible=False), ''

    transform_btn.click(
        fn=on_transform,
        inputs=[working_image, style_selector, strength_slider, seed_input],
        outputs=[output_image, status_box, download_btn, sp_box],
    )

    # QR polling via demo.load — this JS DOES execute on page load
    demo.load(
        fn=None,
        js="""
() => {
    var _last = '';
    setInterval(function() {
        var el = document.querySelector('#result-img img');
        if (!el || !el.src) return;
        var src = el.src;
        if (src.indexOf('/file=') < 0) return;
        if (src === _last) return;
        _last = src;
        var panel = document.getElementById('qr-panel');
        if (!panel) return;
        var qUrl = 'https://api.qrserver.com/v1/create-qr-code/?size=200x200&data=' + encodeURIComponent(src);
        var img = document.createElement('img');
        img.src = qUrl;
        img.style.cssText = 'width:180px;height:180px;border-radius:8px;display:block;margin:0 auto;';
        img.alt = 'QR';
        var p = document.createElement('p');
        p.style.cssText = 'font-size:0.78em;color:#555;margin:6px 0 0;';
        p.textContent = '📱 امسح بالهاتف | Scan with phone';
        panel.innerHTML = '';
        panel.appendChild(img);
        panel.appendChild(p);
    }, 800);
}
"""
    )

demo.queue(max_size=5).launch(share=True, debug=False, show_error=True, allowed_paths=['/tmp'])


---
## 📝 ملاحظات تقنية | Technical Notes

### النماذج المستخدمة | Models Used:
- **InsightFace buffalo_l**: Face detection & embedding
- **IP-Adapter FaceID**: Face identity preservation
- **Counterfeit-V3.0 / Dreamlike Anime**: Style generation (Stable Diffusion)
- **ControlNet Canny**: Structure preservation
- **GFPGAN v1.4**: Face restoration post-processing
- **Real-ESRGAN anime_6B**: 2x upscaling

### البنية التقنية | Pipeline Architecture:
```
Input Photo
    ↓
InsightFace (face detection + embedding)
    ↓
[A] FaceID + ControlNet img2img (primary)
[B] ControlNet img2img only (fallback)
[C] AnimeGAN / OpenCV (guaranteed fallback)
    ↓
GFPGAN face restoration
    ↓
Real-ESRGAN 2x upscale
    ↓
Background preservation
    ↓
Output Styled Image
```

### متطلبات النظام | System Requirements:
- **GPU**: NVIDIA T4 or better (16GB VRAM recommended)
- **RAM**: 12GB+ system memory
- **Storage**: ~5GB for all model weights
- **Runtime**: Google Colab with GPU acceleration

### حدود الاستخدام | Usage Limits:
- **Max image size**: 768×768 px (auto-resized)

---

*🇱🇾 Developed for Libya Tech & IT Day Event · تطوير لفعالية يوم التقنية والمعلومات - ليبيا*